<a href="https://colab.research.google.com/github/kite121/Machine-Learning-Course/blob/main/Lab7_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab-6 : Self-Practice

In this week self-practice, you will implement a neural network model for a regression problem. You will use the [*admission*](./Admission_Predict.csv) dataset attached, used in the previous lab



## 1. Load the dataset and do all the necessary preprocessing

In [ ]:
import torch
import pandas as pd
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split



In [ ]:
data = pd.read_csv("/content/Admission_Predict.csv")
X = data.iloc[:, :-1]
y = data.iloc[:, -1].to_numpy()

In [ ]:
batch_size = 32
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2)


## 2. Create custom pytorch `Dataset`

You should create a class `CustomDataset` that inherits  the abstract class `torch.utils.data.Dataset` from pytorch.

> **Note** You should overwrite `__getitem__`, supporting fetching a data sample for a given key. Subclasses could also optionally overwrite `__len__`, which is expected to return the size of the dataset by many `~torch.utils.data.Sampler` implementations and the default options of `~torch.utils.data.DataLoader`.

#### Split your dataset into train and test data loaders
You can create a `CustomDataset` instance with the entire dataframe and use [`random_split`](https://pytorch.org/docs/stable/data.html#torch.utils.data.random_split) to split it into training and testing datasets. And then, create test and train dataloader. Or you can split using `train_test_split` from sklearn and past the splitted sets to your Custom dataset class.

Create train and test dataloader with `batch_size = 32` each complete the following function

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

class CustumData(Dataset):
    def __init__(self, X, y):
        super().__init__()
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)


    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
# Create the datasets
train_dataset = CustumData(X_train, y_train)
test_dataset = CustumData(X_test, y_test)

# Create the dataloaders
train_dataloader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_dataloader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [ ]:
data, label = next(iter(train_dataloader))
label.shape

torch.Size([32, 1])

## 3. Create the model

Using `nn`, Create a neural network with 1 hidden layers of size 100, each must be followed by a `leaky_relu` activation function and define the forward function

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# complete the code
class Net(nn.Module):
    def __init__(self, n_hidden_unit = 100):
        super(Net, self).__init__()
        self.h1 = nn.Linear(8, 100)
        self.act_func = F.leaky_relu
        self.output = nn.Linear(100, 1)

    def forward(self, x):
        x = self.act_func(self.h1(x))
        x = self.output(x)
        return F.sigmoid(x)


use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")
model_net = Net(n_hidden_unit = 100).to(device)

# 4. Training loop

Define the appropriate loss function and the training loop for the training and the testing dataloader (as done in the lab). Use SGD optimizer with learning rate 0.01 and momentum 0.5

Print the final loss on the test data

In [ ]:
optimizer = torch.optim.AdamW(model_net.parameters())
criterion = nn.MSELoss()
epochs = 100

In [ ]:
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    bar = tqdm(train_loader)
    iteration = 0
    overall_loss = 0

    for data, target in bar:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        iteration += 1
        overall_loss += loss.item()
        bar.set_postfix({"Loss": format(overall_loss / iteration, '.6f')})


def test(model, device, test_loader):
    model.eval()
    test_loss = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            test_loss += loss.item() * data.size(0)

    test_loss /= len(test_loader.dataset)
    print(f"Test set: Average loss: {test_loss:.6f}")

In [ ]:
for epoch in range(1, epochs + 1):
    train(model_net, device, train_dataloader, optimizer, epoch)
    test(model_net, device, test_dataloader)

100%|██████████| 10/10 [00:00<00:00, 309.84it/s, Loss=0.003734]


Test set: Average loss: 0.004773


100%|██████████| 10/10 [00:00<00:00, 314.22it/s, Loss=0.003690]


Test set: Average loss: 0.004602


100%|██████████| 10/10 [00:00<00:00, 299.02it/s, Loss=0.003524]


...

Test set: Average loss: 0.003725


100%|██████████| 10/10 [00:00<00:00, 347.35it/s, Loss=0.001708]


Test set: Average loss: 0.003698


100%|██████████| 10/10 [00:00<00:00, 344.60it/s, Loss=0.001699]


Test set: Average loss: 0.003735


## 5. Compare your Neural network model to a Linear Regression
Train a simple linear regression model on the training set and print MSE on the testing set (`X_test`). Also print the MSE on the test set using the your neural model.

> Compare the results (which performs best) and justify why

In [ ]:
from sklearn.linear_model import Ridge
import sklearn.metrics as metrics
lin_model = Ridge()
lin_model.fit(X_train, y_train)
y_pred = lin_model.predict(X_test)
mse_lin = metrics.mean_squared_error(y_test,  y_pred)
print(mse_lin)


0.004171691286627289
